# Swahili OmniVoice finetune + data-efficiency eval — A100

**Runtime: ~1–1.5 h on a single A100.** Budgets are small (~6.6 h single-speaker data from nb08):
per budget = Stage-0 encode (~3–8 min) + Stage-1 train (5 h ≈ 5 min, 2.5 h ≈ 3 min, 1 h ≈ 1 min) + eval
(~8 min). Plus the zero-shot baseline. Training is fast; Whisper-sw transcription dominates eval.

Mirrors the proven Yorùbá sweep (nb05): hour-budget prefixes of one shuffle, fix-epochs protocol,
full train log streamed to S3. **Swahili is non-tonal**, so the eval is **CER (Whisper-sw) +
voice-fidelity (SSIM) + listen** — no tone metric. Reads the S3 artifacts nb08 produced.
`hf-xet` removed. No edits needed — run top to bottom.

## 1. Install + restart (numpy<2, omnivoice, gate + Whisper; remove hf-xet)

In [ ]:
%pip install --quiet "numpy<2"
%pip install --quiet omnivoice
%pip install --quiet boto3 soundfile soxr librosa speechbrain accelerate safetensors "huggingface_hub>=0.24.0" tqdm matplotlib transformers
%pip uninstall -y hf-xet
import os
print("Installs done. RESTARTING so numpy<2 takes effect — run from the NEXT cell.", flush=True)
os._exit(0)

In [ ]:
import numpy as np
assert np.__version__.startswith("1."), "numpy<2 pin did not take — re-run install + restart"
print("numpy", np.__version__, "OK")

## 2. Clone OmniVoice (verify Swahili code) + finetune surface + STOCK config

In [ ]:
import os, subprocess, sys, shutil, json
OV_DIR="/content/OmniVoice" if os.path.isdir("/content") else "/tmp/OmniVoice"
if os.path.exists(OV_DIR): shutil.rmtree(OV_DIR)
subprocess.run(["git","clone","--depth","1","https://github.com/k2-fsa/OmniVoice.git",OV_DIR],check=True)
OV_EX=os.path.join(OV_DIR,"examples"); sys.path.insert(0,OV_DIR)
assert os.path.exists(os.path.join(OV_EX,"run_finetune.sh")), "MISSING run_finetune.sh — recon stale"
for mod in ["omnivoice.scripts.extract_audio_tokens","omnivoice.cli.train"]:
    r=subprocess.run([sys.executable,"-c",f"import importlib;importlib.import_module('{mod}')"],
                     capture_output=True,text=True,cwd=OV_DIR,env=dict(os.environ,PYTHONPATH=OV_DIR))
    assert r.returncode==0, f"{mod} not importable:\n{r.stderr[-600:]}"
STOCK=json.load(open(os.path.join(OV_EX,"config","train_config_finetune.json")))
assert STOCK.get("init_from_checkpoint")=="k2-fsa/OmniVoice" and STOCK.get("num_audio_codebook")==8
# resolve the Swahili runtime language code from OmniVoice's own map (sw vs swh)
SW_CODE=None
for ln in open(os.path.join(OV_DIR,"docs","lang_id_name_map.tsv")):
    c=ln.rstrip("\n").split("\t")
    if not c: continue
    if c[0] in ("sw","swh") or (len(c)>1 and c[1].lower().startswith("swahili")):
        print("Swahili lang line:",ln.rstrip()); SW_CODE=SW_CODE or c[0]
assert SW_CODE, "Swahili not found in OmniVoice lang map — inspect docs/lang_id_name_map.tsv"
print("finetune surface OK | SW_CODE =",SW_CODE,"| stock codebooks",STOCK.get("num_audio_codebook"))

## 2b. Clone `mosesdaudu001/tone-on-a-budget` (approachB → RewardModels for CER + SSIM)

In [ ]:
# tone-metric package (public) — replaces the old git clone + sys.path of tone_metric/
%pip install --quiet --no-deps "git+https://github.com/mosesdaudu001/tone-on-a-budget.git"
import os, subprocess, sys, shutil, importlib
importlib.invalidate_caches()
import tone_metric
from tone_metric.grpo_reward import RewardModels
CODE_DIR = os.path.dirname(tone_metric.__file__)   # minimal_pairs_draft.json ships as package data
print("tone_metric", tone_metric.__version__, "from", CODE_DIR)


## 3. Secrets + S3 + constants + base snapshot prefetch

In [ ]:
import os, getpass, boto3, random, torch, numpy as np, shutil
def _secret(k):
    try:
        from google.colab import userdata
        v=userdata.get(k)
        if v: return v
    except Exception: pass
    return os.environ.get(k) or getpass.getpass(f"{k}: ")
os.environ["AWS_ACCESS_KEY_ID"]=_secret("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"]=_secret("AWS_SECRET_ACCESS_KEY")
os.environ["AWS_DEFAULT_REGION"]=os.environ.get("AWS_DEFAULT_REGION","us-east-1")
HF_TOKEN=_secret("HF_TOKEN"); os.environ["HF_TOKEN"]=HF_TOKEN or ""
if HF_TOKEN:
    from huggingface_hub import login; login(token=HF_TOKEN)
os.environ["HF_HUB_DISABLE_XET"]="1"; os.environ["HF_HUB_ENABLE_HF_TRANSFER"]="0"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"]="60"; os.environ["HF_HUB_ETAG_TIMEOUT"]="60"
try:
    import huggingface_hub.constants as _hc; _hc.HF_HUB_DISABLE_XET=True
except Exception: pass
BUCKET="codec-audio-data"
s3=boto3.client("s3",region_name=os.environ["AWS_DEFAULT_REGION"]); s3.head_bucket(Bucket=BUCKET)
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
assert DEVICE=="cuda", "the sweep trains — needs the A100"
SEED=4242; SR=24000; LANG_CODE=SW_CODE
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ===================== SWEEP KNOBS =====================
SWEEP_BUDGETS  = [5.0, 2.5, 1.0]   # only ~6.6h single-speaker collected by nb08; + 0h zero-shot baseline
EPOCHS_TARGET  = 10
CLIPS_PER_STEP = 41.3          # estimate (OpenBible verses ~ Yoruba bible clip length); training is cheap
BATCH_TOKENS   = 16384
DEV_HOURS      = 0.3
# =======================================================

SW_MANIFEST="tts_data/swahili_gold/openbible_sw.v1.jsonl"
HOLDOUTS_KEY="tts_data/swahili_gold/holdouts.v1.json"
REF_KEY="tts_data/swahili_gold/ref.wav"
ABLATION_KEY="tts_data/swahili/omnivoice_ablation/data_efficiency_sw.v1.json"
CURVE_KEY="tts_data/swahili/omnivoice_ablation/curve_sw.png"
WORK="/content/sw_sweep" if os.path.isdir("/content") else "/tmp/sw_sweep"
LOCAL_WAVS=os.path.join(WORK,"wavs"); os.makedirs(LOCAL_WAVS,exist_ok=True)

from huggingface_hub import snapshot_download
HUB=os.path.join(os.path.expanduser(os.environ.get("HF_HOME","~/.cache/huggingface")),"hub")
if os.path.isdir(os.path.join(HUB,".locks")): shutil.rmtree(os.path.join(HUB,".locks"),ignore_errors=True)
OV_BASE=None
for _k in range(3):
    try: OV_BASE=snapshot_download("k2-fsa/OmniVoice",max_workers=1,etag_timeout=60); break
    except Exception as e: print("base prefetch retry:",type(e).__name__,e)
for _k in range(3):
    try: snapshot_download("eustlb/higgs-audio-v2-tokenizer",max_workers=1,etag_timeout=60); break
    except Exception as e: print("codec prefetch retry:",type(e).__name__,e)
assert OV_BASE and os.path.isdir(os.path.join(OV_BASE,"audio_tokenizer")), "base prefetch failed"
print("base snapshot:",OV_BASE,"| budgets:",SWEEP_BUDGETS)

## 4. Data load — download wavs for the largest budget + dev (once)

In [ ]:
import io, json, unicodedata
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm
def norm_key(s): return " ".join(unicodedata.normalize("NFC",str(s)).lower().split())
hold=json.loads(s3.get_object(Bucket=BUCKET,Key=HOLDOUTS_KEY)["Body"].read())
HELD=set(norm_key(e["text"]) for e in hold["eval_texts"])
cands=[]
for raw in io.TextIOWrapper(s3.get_object(Bucket=BUCKET,Key=SW_MANIFEST)["Body"],encoding="utf-8"):
    r=json.loads(raw)
    if norm_key(r["text"]) in HELD: continue
    cands.append(dict(id=r["id"],text=r["text"],dur=float(r["duration_sec"]),wav_key=r["audio_s3_key"]))
rng=random.Random(SEED); rng.shuffle(cands)
def take_hours(rows,budget_h):
    if budget_h is None: return rows,sum(c["dur"] for c in rows)
    acc,out=0.0,[]
    for c in rows:
        if acc>=budget_h*3600: break
        out.append(c); acc+=c["dur"]
    return out,acc
dev_rows,dev_s=take_hours(cands,DEV_HOURS); rest=cands[len(dev_rows):]
print(f"candidates {len(cands)} ({sum(c['dur'] for c in cands)/3600:.2f} h) | dev {len(dev_rows)} | rest {len(rest)}")
# recalibrate CLIPS_PER_STEP from actual mean duration (batch_tokens budget is audio-dominated -> clips/step
# scales ~ 1/duration). Yoruba ref: 41.3 clips/step at 9.56 s/clip (28.5h/10731 clips). Keeps ~epochs honest.
avg_dur=float(np.mean([c["dur"] for c in cands])); CLIPS_PER_STEP=41.3*(9.56/max(avg_dur,1e-6))
print(f"avg clip {avg_dur:.2f}s -> CLIPS_PER_STEP {CLIPS_PER_STEP:.1f} (Yoruba ref 9.56s / 41.3)")
largest=max(SWEEP_BUDGETS); big_train,_=take_hours(rest,largest); need=big_train+dev_rows
def _dl(c):
    local=os.path.join(LOCAL_WAVS,c["id"]+".wav")
    if not os.path.exists(local): s3.download_file(BUCKET,c["wav_key"],local)
    c["audio_path"]=local; return c
with ThreadPoolExecutor(max_workers=32) as ex:
    list(tqdm(ex.map(_dl,need),total=len(need),desc=f"download wavs (<= {largest:g}h + dev)"))
ID2PATH={c["id"]:c["audio_path"] for c in need}
print("wavs local:",len(ID2PATH))

## 5. Eval stack — Whisper-sw (CER) + ECAPA (SSIM) + probe set + ref voice

In [ ]:
import json, soundfile as sf, soxr, numpy as np, re
from transformers import pipeline
rm=RewardModels(device=DEVICE); assert rm.ecapa is not None, "ECAPA needed for SSIM"
asr_pipe=pipeline("automatic-speech-recognition", model="openai/whisper-large-v3",
                  device=0, torch_dtype=torch.float16, chunk_length_s=30)
def _norm(t):
    t=str(t).lower(); t=re.sub(r"[^\w\s]"," ",t,flags=re.UNICODE); return " ".join(t.split())
def asr_sw(wav,fs):
    w=np.asarray(wav,dtype="float32"); w16=soxr.resample(w,fs,16000) if fs!=16000 else w
    out=asr_pipe({"array":w16,"sampling_rate":16000},
                 generate_kwargs={"language":"swahili","task":"transcribe"})
    return out["text"]
# probe = held-out Swahili eval texts (no minimal pairs — non-tonal)
probe_lines=[dict(id=e["base"],text=e["text"]) for e in hold["eval_texts"]]
REF_WAV_PATH=f"{WORK}/ref.wav"; s3.download_file(BUCKET,REF_KEY,REF_WAV_PATH)
try: REF_TEXT=json.loads(s3.get_object(Bucket=BUCKET,Key=REF_KEY+".json")["Body"].read())["text"]
except Exception: REF_TEXT=probe_lines[0]["text"]
ref_wav,ref_sr=sf.read(REF_WAV_PATH,dtype="float32")
if ref_wav.ndim>1: ref_wav=ref_wav.mean(-1)
print("eval stack ready | probe lines",len(probe_lines),"| ref text:",REF_TEXT[:50])

## 6. Helpers (train + eval)

In [ ]:
import os, json, subprocess, sys, glob, datetime, shutil, gc
import soundfile as sf, numpy as np
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm
import IPython.display as ipd, torch
from omnivoice import OmniVoice

AUX=["tokenizer.json","tokenizer_config.json","special_tokens_map.json","vocab.json","merges.txt",
     "chat_template.jinja","generation_config.json","added_tokens.json"]
def materialize_ckpt(d):
    have=set(os.listdir(d))
    for fn in AUX:
        if fn not in have and os.path.exists(os.path.join(OV_BASE,fn)): shutil.copy(os.path.join(OV_BASE,fn),os.path.join(d,fn))
    if not os.path.isdir(os.path.join(d,"audio_tokenizer")): shutil.copytree(os.path.join(OV_BASE,"audio_tokenizer"),os.path.join(d,"audio_tokenizer"))
    return d
def write_manifest(rows,path):
    with open(path,"w",encoding="utf-8") as f:
        for c in rows:
            f.write(json.dumps({"id":c["id"],"audio_path":ID2PATH[c["id"]],"text":c["text"],
                                "language_id":LANG_CODE,"duration":round(c["dur"],3)},ensure_ascii=False)+"\n")
def stage0(split,split_jsonl,tag,token_dir):
    out_tar=os.path.join(token_dir,split,"audios","shard-%06d.tar")
    out_jsonl=os.path.join(token_dir,split,"txts","shard-%06d.jsonl")
    os.makedirs(os.path.dirname(out_tar),exist_ok=True); os.makedirs(os.path.dirname(out_jsonl),exist_ok=True)
    cmd=[sys.executable,"-m","omnivoice.scripts.extract_audio_tokens","--input_jsonl",split_jsonl,
         "--tar_output_pattern",out_tar,"--jsonl_output_pattern",out_jsonl,
         "--tokenizer_path","eustlb/higgs-audio-v2-tokenizer","--nj_per_gpu","3","--shuffle","True"]
    subprocess.run(cmd,check=True,cwd=OV_DIR,env=dict(os.environ,CUDA_VISIBLE_DEVICES="0",PYTHONPATH=OV_DIR))
    lst=os.path.join(token_dir,split,"data.lst"); assert os.path.exists(lst), f"Stage0 no {lst}"
    return lst
def build_cfg(steps,train_lst,dev_lst,tag,output_dir):
    save_eval=max(50,steps//8)
    cfg=dict(STOCK); cfg.update({
        "init_from_checkpoint":"k2-fsa/OmniVoice","resume_from_checkpoint":None,"attn_implementation":"sdpa",
        "learning_rate":1e-5,"steps":int(steps),"warmup_type":"ratio","warmup_ratio":0.01,"warmup_steps":0,
        "batch_tokens":BATCH_TOKENS,"gradient_accumulation_steps":1,"mixed_precision":"bf16","allow_tf32":True,
        "num_workers":2,"logging_steps":max(10,save_eval//5),"eval_steps":save_eval,"save_steps":save_eval,
        "keep_last_n_checkpoints":-1,"seed":SEED,"language_ratio":1.0})
    for k,v in {"max_sample_tokens":2000,"min_sample_tokens":50,"max_batch_size":64}.items(): cfg.setdefault(k,v)
    data_cfg={"train":[{"manifest_path":[train_lst]}],"dev":[{"manifest_path":[dev_lst]}]}
    cp=os.path.join(WORK,f"train_config_{tag}.json"); dp=os.path.join(WORK,f"data_config_{tag}.json")
    json.dump(cfg,open(cp,"w"),indent=2); json.dump(data_cfg,open(dp,"w"),indent=2)
    return cp,dp,cfg
def stage1(cp,dp,output_dir,tag):
    cmd=["accelerate","launch","--gpu_ids","0","--num_processes","1","-m","omnivoice.cli.train",
         "--train_config",cp,"--data_config",dp,"--output_dir",output_dir]
    print(">>"," ".join(cmd)); logf=os.path.join(WORK,f"train_{tag}.log")
    proc=subprocess.Popen(cmd,cwd=OV_DIR,env=dict(os.environ,PYTHONPATH=OV_DIR),
                          stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    with open(logf,"w") as lf:
        for line in proc.stdout:
            lf.write(line); low=line.lower()
            if ("train/loss" in line) or ("eval loss" in low) or ("saved checkpoint" in low) \
               or ("error" in low) or ("traceback" in low): print(line,end="")
    rc=proc.wait(); s3.upload_file(logf,BUCKET,f"tts_data/swahili/sweep_logs/train_{tag}.log")
    assert rc==0, f"training exited {rc} — see s3 sweep_logs/train_{tag}.log"
    ck=sorted(glob.glob(os.path.join(output_dir,"checkpoint-*")),key=lambda p:int(p.rsplit("-",1)[-1]))
    assert ck, "no checkpoint written"; return ck[-1]

def score_clip(wav,fs,text):
    hyp=asr_sw(wav,fs)
    return dict(cer=RewardModels.cer(_norm(text),_norm(hyp)),
                ssim=rm.ssim(wav,fs,ref_wav,ref_sr),
                len_ratio=(len(wav)/fs)/max(0.06*len(text),1e-6),wav=wav,hyp=hyp)
def eval_model(m,tag,hours):
    torch.manual_seed(SEED); rows=[]
    for p in tqdm(probe_lines,desc=f"eval[{tag}]"):
        a=m.generate(text=p["text"],language=LANG_CODE,ref_audio=REF_WAV_PATH,ref_text=REF_TEXT)
        rows.append(dict(**score_clip(np.asarray(a[0],dtype="float32"),SR,p["text"])))
    v=lambda k:[r[k] for r in rows if r[k]==r[k]]
    AGG=dict(n=len(rows),cer=float(np.mean(v("cer"))),ssim=float(np.mean(v("ssim"))),
             len_ratio=float(np.mean(v("len_ratio"))))
    print(f"[{tag}] {hours:>5.1f}h  CER {AGG['cer']:.3f}  SSIM {AGG['ssim']:.3f}  (n={AGG['n']})")
    pre=f"tts_data/swahili/omnivoice_ft_probe/{tag}"
    for j,r in enumerate(rows[:2]):
        loc=f"{WORK}/ev_{tag}_{j}.wav"; sf.write(loc,r["wav"],SR); s3.upload_file(loc,BUCKET,f"{pre}/{j}.wav")
        print("   hyp:",r["hyp"][:64]); ipd.display(ipd.Audio(r["wav"],rate=SR))
    return AGG
def append_table(tag,hours,train_hours,clips,steps,AGG):
    try: TABLE=json.loads(s3.get_object(Bucket=BUCKET,Key=ABLATION_KEY)["Body"].read())
    except Exception:
        TABLE={"meta":{"axis":"data-efficiency (clean Swahili hours -> CER/SSIM)","model":"k2-fsa/OmniVoice finetune",
                       "lang":LANG_CODE,"protocol":f"fix-epochs={EPOCHS_TARGET} @ batch_tokens={BATCH_TOKENS}","seed":SEED},"runs":{}}
    TABLE["runs"][tag]={"hours_budget":hours,"train_hours_actual":round(train_hours,3),"train_clips":clips,
        "steps":steps,"cer":AGG["cer"],"ssim":AGG["ssim"],"len_ratio":AGG["len_ratio"],
        "date":datetime.date.today().isoformat(),"ckpt_s3":f"s3://{BUCKET}/tts_checkpoints/omnivoice_swahili/{tag}/"}
    s3.put_object(Bucket=BUCKET,Key=ABLATION_KEY,Body=json.dumps(TABLE,ensure_ascii=False,indent=2).encode())
    return TABLE
def upload_ckpt(final_ckpt,tag):
    pre=f"tts_checkpoints/omnivoice_swahili/{tag}"; files=[]
    for root,_,fns in os.walk(final_ckpt):
        for fn in fns:
            lp=os.path.join(root,fn); files.append((lp,f"{pre}/{os.path.relpath(lp,final_ckpt)}"))
    with ThreadPoolExecutor(max_workers=16) as ex:
        list(ex.map(lambda t: s3.upload_file(t[0],BUCKET,t[1]),files))
    print(f"  uploaded {len(files)} ckpt files -> s3://{BUCKET}/{pre}/")
def run_one_budget(hours):
    tag=f"{hours:g}h"; print(f"\n################  BUDGET {tag}  ################")
    train_rows,train_s=take_hours(rest,hours); assert train_rows
    token_dir=os.path.join(WORK,"tokens",tag); output_dir=os.path.join(WORK,"exp",tag); os.makedirs(output_dir,exist_ok=True)
    tj=os.path.join(WORK,f"train_{tag}.jsonl"); dj=os.path.join(WORK,f"dev_{tag}.jsonl")
    write_manifest(train_rows,tj); write_manifest(dev_rows,dj)
    steps=max(50,round(EPOCHS_TARGET*len(train_rows)/CLIPS_PER_STEP))
    print(f"{tag}: {len(train_rows)} clips / {train_s/3600:.2f} h -> {steps} steps (~{EPOCHS_TARGET} ep)")
    train_lst=stage0("train",tj,tag,token_dir); dev_lst=stage0("dev",dj,tag,token_dir)
    cp,dp,cfg=build_cfg(steps,train_lst,dev_lst,tag,output_dir)
    final_ckpt=stage1(cp,dp,output_dir,tag); upload_ckpt(final_ckpt,tag)
    m=OmniVoice.from_pretrained(materialize_ckpt(final_ckpt),device_map="cuda:0",dtype=torch.float16)
    AGG=eval_model(m,tag,hours); del m; gc.collect(); torch.cuda.empty_cache()
    append_table(tag,hours,train_s/3600,len(train_rows),steps,AGG)
    print(f"################  {tag} DONE: CER={AGG['cer']:.3f} SSIM={AGG['ssim']:.3f}  ################")
    return AGG
print("helpers defined.")

## 7. Zero-shot baseline (0 h)

In [ ]:
import torch, gc
from omnivoice import OmniVoice
m0=OmniVoice.from_pretrained(OV_BASE,device_map="cuda:0",dtype=torch.float16)
AGG0=eval_model(m0,"base",0.0); del m0; gc.collect(); torch.cuda.empty_cache()
append_table("base",0.0,0.0,0,0,AGG0)
print("zero-shot Swahili:",AGG0)

## 8. Budget 5 h

In [ ]:
A5=run_one_budget(5.0)

## 9. Budget 2.5 h

In [ ]:
A25=run_one_budget(2.5)

## 10. Budget 1 h

In [ ]:
A1=run_one_budget(1.0)

## 11. Curve — CER & SSIM vs hours

In [ ]:
import json, numpy as np, matplotlib.pyplot as plt
TABLE=json.loads(s3.get_object(Bucket=BUCKET,Key=ABLATION_KEY)["Body"].read())
pts=sorted([(r["train_hours_actual"],r["cer"],r["ssim"]) for r in TABLE["runs"].values()],key=lambda x:x[0])
xs=[p[0] for p in pts]; cer=[p[1] for p in pts]; ss=[p[2] for p in pts]
print(f"{'hours':>7} {'CER':>7} {'SSIM':>7}")
for h,c,s_ in pts: print(f"{h:>7.2f} {c:>7.3f} {s_:>7.3f}")
fig,ax1=plt.subplots(figsize=(6.6,4.2))
ax1.plot(xs,cer,"o-",color="crimson",label="CER"); ax1.set_xlabel("clean Swahili hours"); ax1.set_ylabel("CER",color="crimson")
ax2=ax1.twinx(); ax2.plot(xs,ss,"s--",color="navy",label="SSIM"); ax2.set_ylabel("SSIM (voice fidelity)",color="navy")
ax1.set_title("Swahili (non-tonal): the recipe generalizes"); fig.tight_layout()
plt.savefig(f"{WORK}/curve_sw.png",dpi=130); s3.upload_file(f"{WORK}/curve_sw.png",BUCKET,CURVE_KEY); plt.show()
print("-> s3://%s/%s"%(BUCKET,CURVE_KEY))

### Read-out
- **CER** ≈ flat-low across budgets (Swahili is already intelligible zero-shot) — the *expected* contrast
  with Yorùbá, whose tone curve *rises* with data. Story: **data-efficiency depends on what is broken
  zero-shot** — a tonal language needs data to install tone; a non-tonal one is already fine.
- **SSIM** = voice fidelity to the single OpenBible narrator; should stay high (recipe preserves voice).
- The **ear** (inline clips + `…/omnivoice_ft_probe/{tag}/`) is the decisive quality check.
- Artifacts: `data_efficiency_sw.v1.json` + `curve_sw.png` on S3; checkpoints under `omnivoice_swahili/{tag}/`.